# FIT5196 Assessment 1 - Solution Notebook

**Group:** Group050


**Members:** <br>
Yu Wang ID:  ; <br>
Xingao Zhan ID:   ;  <br>
Keshu Zhang ID:   ; <br>
Qingzhuo Zhao ID: 26662841

Rename this file to `GroupNNN_solution.ipynb`. Replace every `NNN` placeholder,
complete the investigation and remove instructional prompts that are no longer
useful. This template supplies structure only; it does not contain assessed
source paths, transformations or answers.


## 0. Configuration and reproducibility

Keep all configurable paths in this section. The final notebook must run with
**Restart and Run All** without manual file edits or network access.


In [ ]:
from pathlib import Path

GROUP_ID = "Group050"
INPUT_DIR = Path("raw_input")
OUTPUT_DIR = Path("outputs")
TEMPLATE_DIR = Path("templates")

if not TEMPLATE_DIR.exists() and Path.cwd().name == "templates":
    TEMPLATE_DIR = Path(".")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


### 0.1 Environment and dependencies

Import the libraries used by your submitted workflow. Record non-standard
dependencies in `requirements.txt`.


In [ ]:
# EVIDENCE: SEC-0.1-ENVIRONMENT
import json
import platform
import sys
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd

def show(frame, title=None):
    if title:
        print(f"\n{title}")
    print(frame.to_string(index=False))

environment = pd.DataFrame(
    [
        {"component": "Python", "version": platform.python_version()},
        {"component": "pandas", "version": pd.__version__},
        {"component": "NumPy", "version": np.__version__},
        {"component": "XML parser", "version": "xml.etree.ElementTree (stdlib)"},
        {"component": "JSON parser", "version": "json (stdlib)"},
    ]
)
show(environment, "Environment and dependencies")



Environment and dependencies
  component                        version
     Python                        3.12.13
     pandas                          2.2.3
      NumPy                          2.0.2
 XML parser xml.etree.ElementTree (stdlib)
JSON parser                  json (stdlib)


## 1. Parse and profile the two sources

Use structured JSON and XML parsers. Record source grains, nested/repeated
structures, candidate keys, formats, missing-value conventions and evidence of
within-source or cross-source overlap.


### 1.1 JSON structure and profile


In [11]:
# EVIDENCE: SEC-1.1-JSON-PROFILE
JSON_PATH = INPUT_DIR / "input.json"
with JSON_PATH.open(encoding="utf-8") as handle:
    json_source = json.load(handle)

json_customers = json_source["customerProfiles"]
json_orders = json_source["orders"]
json_items = [item for order in json_orders for item in order["shoppingCart"]]
json_deliveries = [order["delivery"] for order in json_orders]
json_reviews = json_source["productReviews"]

def profile_records(source_collection, grain, records, key_field):
    keys = [record.get(key_field) for record in records]
    non_missing = [key for key in keys if key not in (None, "")]
    frequencies = Counter(non_missing)
    return {
        "source_collection": source_collection,
        "grain": grain,
        "candidate_key": key_field,
        "rows": len(records),
        "missing_key": len(keys) - len(non_missing),
        "unique_key": len(frequencies),
        "duplicate_key_groups": sum(count > 1 for count in frequencies.values()),
        "duplicate_extra_rows": len(non_missing) - len(frequencies),
    }

json_profile = pd.DataFrame(
    [
        profile_records("customerProfiles[]", "one customer profile", json_customers, "customerID"),
        profile_records("orders[]", "one source order", [order["header"] for order in json_orders], "orderID"),
        profile_records("orders[].shoppingCart[]", "one source order item", json_items, "orderItemID"),
        profile_records("orders[].delivery", "one source delivery", json_deliveries, "deliveryID"),
        profile_records("productReviews[]", "one source review", json_reviews, "reviewID"),
    ]
)
show(json_profile, "JSON collection profile")

json_structure = pd.DataFrame(
    [
        {"path": "customerProfiles[]", "representation": "repeated root array", "fields": ", ".join(json_customers[0].keys())},
        {"path": "orders[]", "representation": "repeated root array with header, shoppingCart[] and delivery", "fields": ", ".join(json_orders[0].keys())},
        {"path": "orders[].header", "representation": "nested object", "fields": ", ".join(json_orders[0]["header"].keys())},
        {"path": "orders[].shoppingCart[]", "representation": "nested repeated array", "fields": ", ".join(json_items[0].keys())},
        {"path": "orders[].delivery", "representation": "nested object", "fields": ", ".join(json_deliveries[0].keys())},
        {"path": "productReviews[]", "representation": "repeated root array", "fields": ", ".join(json_reviews[0].keys())},
    ]
)
show(json_structure, "JSON nesting and fields")

first_header = json_orders[0]["header"]
first_delivery = json_orders[0]["delivery"]
json_formats = pd.DataFrame(
    [
        {"concept": "timestamp", "path": "orders[].header.orderTimestamp", "example": repr(first_header["orderTimestamp"]), "Python type": type(first_header["orderTimestamp"]).__name__},
        {"concept": "date", "path": "orders[].delivery.dispatchDate", "example": repr(first_delivery["dispatchDate"]), "Python type": type(first_delivery["dispatchDate"]).__name__},
        {"concept": "boolean", "path": "orders[].header.expeditedDelivery", "example": repr(first_header["expeditedDelivery"]), "Python type": type(first_header["expeditedDelivery"]).__name__},
        {"concept": "currency value", "path": "orders[].header.orderPrice", "example": repr(first_header["orderPrice"]), "Python type": type(first_header["orderPrice"]).__name__},
        {"concept": "percentage points", "path": "orders[].header.couponDiscount", "example": repr(first_header["couponDiscount"]), "Python type": type(first_header["couponDiscount"]).__name__},
        {"concept": "missing optional string", "path": "orders[].header.couponCode", "example": "empty string count=" + str(sum(order["header"].get("couponCode") == "" for order in json_orders)), "Python type": "str"},
    ]
)
show(json_formats, "JSON source-format evidence")

FileNotFoundError: [Errno 2] No such file or directory: 'raw_input/input.json'

### 1.2 XML structure and profile


In [12]:
# EVIDENCE: SEC-1.2-XML-PROFILE
xml_tree = ET.parse(XML_PATH)
xml_root = xml_tree.getroot()

xml_orders = xml_root.findall("./Orders/Order")
xml_headers = [order.find("Header") for order in xml_orders]
xml_items = xml_root.findall("./Orders/Order/Shopping_Cart/Item")
xml_deliveries = xml_root.findall("./Orders/Order/Delivery")
xml_products = xml_root.findall("./ProductCatalogue/Product")
xml_reviews = xml_root.findall("./ProductReviews/Review")
xml_warehouses = list(xml_root.find("WarehouseDirectory"))

def element_records(elements):
    return [{child.tag: child.text or "" for child in element} for element in elements]

xml_header_records = element_records(xml_headers)
xml_item_records = element_records(xml_items)
xml_delivery_records = element_records(xml_deliveries)
xml_product_records = element_records(xml_products)
xml_review_records = element_records(xml_reviews)
xml_warehouse_records = element_records(xml_warehouses)

xml_profile = pd.DataFrame(
    [
        profile_records("/OperationsExport/Orders/Order/Header", "one source order", xml_header_records, "Order_ID"),
        profile_records("/OperationsExport/Orders/Order/Shopping_Cart/Item", "one source order item", xml_item_records, "Order_Item_ID"),
        profile_records("/OperationsExport/Orders/Order/Delivery", "one source delivery", xml_delivery_records, "Delivery_ID"),
        profile_records("/OperationsExport/ProductCatalogue/Product", "one product", xml_product_records, "Product_ID"),
        profile_records("/OperationsExport/ProductReviews/Review", "one source review", xml_review_records, "Review_ID"),
        profile_records("/OperationsExport/WarehouseDirectory/*", "one warehouse reference", xml_warehouse_records, "Name"),
    ]
)
show(xml_profile, "XML collection profile")

xml_structure = pd.DataFrame(
    [
        {"path": "/OperationsExport", "representation": "root element", "child elements": ", ".join(child.tag for child in xml_root)},
        {"path": "/OperationsExport/Orders/Order", "representation": "repeated element with Header, Shopping_Cart and Delivery", "child elements": ", ".join(child.tag for child in xml_orders[0])},
        {"path": "/OperationsExport/Orders/Order/Shopping_Cart/Item", "representation": "nested repeated element", "child elements": ", ".join(xml_item_records[0].keys())},
        {"path": "/OperationsExport/ProductCatalogue/Product", "representation": "repeated element", "child elements": ", ".join(xml_product_records[0].keys())},
        {"path": "/OperationsExport/ProductReviews/Review", "representation": "repeated element", "child elements": ", ".join(xml_review_records[0].keys())},
        {"path": "/OperationsExport/WarehouseDirectory/*", "representation": "source-specific reference collection", "child elements": ", ".join(xml_warehouse_records[0].keys())},
    ]
)
show(xml_structure, "XML nesting and fields")

first_xml_header = xml_header_records[0]
first_xml_delivery = xml_delivery_records[0]
xml_formats = pd.DataFrame(
    [
        {"concept": "timestamp", "path": ".../Header/Order_Timestamp", "example": repr(first_xml_header["Order_Timestamp"]), "XML representation": "text"},
        {"concept": "date", "path": ".../Delivery/Dispatch_Date", "example": repr(first_xml_delivery["Dispatch_Date"]), "XML representation": "text"},
        {"concept": "boolean", "path": ".../Header/Expedited_Delivery", "example": repr(first_xml_header["Expedited_Delivery"]), "XML representation": "Y/N text"},
        {"concept": "currency value", "path": ".../Header/Order_Price", "example": repr(first_xml_header["Order_Price"]), "XML representation": "currency-labelled text"},
        {"concept": "percentage", "path": ".../Header/Coupon_Discount", "example": repr(first_xml_header["Coupon_Discount"]), "XML representation": "percent-labelled text"},
        {"concept": "missing optional string", "path": ".../Header/Coupon_Code", "example": "empty element count=" + str(sum(record["Coupon_Code"] == "" for record in xml_header_records)), "XML representation": "empty element"},
    ]
)
show(xml_formats, "XML source-format evidence")


NameError: name 'XML_PATH' is not defined

### 1.3 Source comparison and assumptions


In [13]:
# EVIDENCE: SEC-1.3-SOURCE-COMPARISON
json_key_sets = {
    "orders": {record["orderID"] for record in [order["header"] for order in json_orders]},
    "order_items": {record["orderItemID"] for record in json_items},
    "deliveries": {record["deliveryID"] for record in json_deliveries},
    "product_reviews": {record["reviewID"] for record in json_reviews},
}
xml_key_sets = {
    "orders": {record["Order_ID"] for record in xml_header_records},
    "order_items": {record["Order_Item_ID"] for record in xml_item_records},
    "deliveries": {record["Delivery_ID"] for record in xml_delivery_records},
    "product_reviews": {record["Review_ID"] for record in xml_review_records},
}

overlap_rows = []
for entity in json_key_sets:
    json_keys = json_key_sets[entity]
    xml_keys = xml_key_sets[entity]
    overlap = json_keys & xml_keys
    overlap_rows.append(
        {
            "entity": entity,
            "JSON unique keys": len(json_keys),
            "XML unique keys": len(xml_keys),
            "cross-source overlap": len(overlap),
            "JSON only": len(json_keys - xml_keys),
            "XML only": len(xml_keys - json_keys),
            "union before field reconciliation": len(json_keys | xml_keys),
        }
    )
overlap_profile = pd.DataFrame(overlap_rows)
show(overlap_profile, "Cross-source key overlap")

combined_repeat_profile = pd.concat(
    [
        json_profile.assign(source="JSON"),
        xml_profile.assign(source="XML"),
    ],
    ignore_index=True,
)[
    ["source", "source_collection", "grain", "candidate_key", "rows", "missing_key", "unique_key", "duplicate_key_groups", "duplicate_extra_rows"]
]
show(combined_repeat_profile, "Within-source repeat evidence")

source_coverage = pd.DataFrame(
    [
        {"target entity": "orders", "JSON evidence": "orders[].header", "XML evidence": "/OperationsExport/Orders/Order/Header", "coverage": "both"},
        {"target entity": "order_items", "JSON evidence": "orders[].shoppingCart[]", "XML evidence": "/OperationsExport/Orders/Order/Shopping_Cart/Item", "coverage": "both"},
        {"target entity": "customers", "JSON evidence": "customerProfiles[]", "XML evidence": "no full customer collection", "coverage": "JSON only"},
        {"target entity": "deliveries", "JSON evidence": "orders[].delivery", "XML evidence": "/OperationsExport/Orders/Order/Delivery", "coverage": "both"},
        {"target entity": "products", "JSON evidence": "product IDs only in related records", "XML evidence": "/OperationsExport/ProductCatalogue/Product", "coverage": "XML only for full entity"},
        {"target entity": "product_reviews", "JSON evidence": "productReviews[]", "XML evidence": "/OperationsExport/ProductReviews/Review", "coverage": "both"},
        {"target entity": "warehouse reference", "JSON evidence": "warehouse name in order header", "XML evidence": "/OperationsExport/WarehouseDirectory/*", "coverage": "source-specific helper"},
    ]
)
show(source_coverage, "Source coverage and source-specific collections")

key_relationships = pd.DataFrame(
    [
        {"entity / grain": "orders / one order", "candidate primary key": "order_id", "candidate foreign keys": "customer_id -> customers.customer_id", "source evidence": "header customer and order identifiers in both sources"},
        {"entity / grain": "order_items / one line item", "candidate primary key": "order_item_id", "candidate foreign keys": "order_id -> orders.order_id | product_id -> products.product_id", "source evidence": "nested shopping-cart identifiers in both sources"},
        {"entity / grain": "customers / one customer", "candidate primary key": "customer_id", "candidate foreign keys": "none", "source evidence": "customerProfiles[].customerID"},
        {"entity / grain": "deliveries / one delivery", "candidate primary key": "delivery_id", "candidate foreign keys": "order_id -> orders.order_id", "source evidence": "nested delivery identifiers in both sources"},
        {"entity / grain": "products / one product", "candidate primary key": "product_id", "candidate foreign keys": "none in the six-table target contract", "source evidence": "/OperationsExport/ProductCatalogue/Product/Product_ID"},
        {"entity / grain": "product_reviews / one review", "candidate primary key": "review_id", "candidate foreign keys": "order_id -> orders.order_id | order_item_id -> order_items.order_item_id | product_id -> products.product_id | customer_id -> customers.customer_id", "source evidence": "review identifiers in both sources"},
        {"entity / grain": "warehouse reference / one warehouse", "candidate primary key": "Name", "candidate foreign keys": "referenced by orders.nearest_warehouse", "source evidence": "/OperationsExport/WarehouseDirectory/*/Name"},
    ]
)
show(key_relationships, "Candidate keys and relationships to verify after reconciliation")

assumptions = pd.DataFrame(
    [
        {"assumption_id": "ASM-01", "decision before transformation": "Use the published table primary key as the stable business key at each target grain; do not invent identifiers."},
        {"assumption_id": "ASM-02", "decision before transformation": "Normalise comparable types and strings before comparing duplicates or cross-source overlap."},
        {"assumption_id": "ASM-03", "decision before transformation": "Retain one canonical row when all normalised non-missing values agree; record any disagreement as validation evidence rather than applying source precedence."},
        {"assumption_id": "ASM-04", "decision before transformation": "Preserve identifier leading zeros and source case; do not lower-case structured categories."},
        {"assumption_id": "ASM-05", "decision before transformation": "Parse JSON/XML structurally before applying regex to bounded narrative fields."},
        {"assumption_id": "ASM-06", "decision before transformation": "Use literal NaN only for prescribed missing string outputs; never substitute it into required numeric, boolean, primary-key or foreign-key fields."},
        {"assumption_id": "ASM-07", "decision before transformation": "Recompute line and order arithmetic in the published sequence and use reported source amounts only as validation evidence."},
        {"assumption_id": "ASM-08", "decision before transformation": "Flatten repeated items only into their target one-to-many table; do not flatten all entities into one wide table."},
        {"assumption_id": "ASM-09", "decision before transformation": "Treat the observed counts as profiling evidence only; derive every output and validation result from the current allocated files."},
    ]
)
show(assumptions, "Pre-transformation assumption register")

assert combined_repeat_profile["missing_key"].sum() == 0, "A candidate source key is missing"
print("\nTask 1 source-profile checks: PASS")


NameError: name 'json_orders' is not defined

## 2. Source-to-target mapping

The supplied CSV contains one row for every required target field. Complete
only the blank columns. It is a field-lineage record, not a copy of the data.

| Column | What to enter |
|---|---|
| `source_format` | `JSON`, `XML`, `both`, or `derived` |
| `json_source_path` | Structural JSON path, or blank when not applicable |
| `xml_source_path` | Structural XML path, or blank when not applicable |
| `transformation_or_derivation` | Type conversion, cleaning rule, formula, or direct-copy rule |
| `overlap_or_conflict_rule` | How duplicates/overlap are compared and what happens on disagreement |
| `notebook_evidence` | A stable section name or code-cell label in this notebook |

Use ` | ` to separate several source fields used in one derivation. These
fictional examples explain the level of detail without revealing an assessed
path:

| Target | Source paths | Transformation | Overlap/conflict rule |
|---|---|---|---|
| `example.customer_id` | JSON `customers[].id`; XML `/customers/customer/@id` | Trim surrounding whitespace; preserve leading zeros | After normalisation, require the two non-missing values to agree; otherwise record a validation failure |
| `example.line_revenue` | JSON `orders[].items[].qty | orders[].items[].price`; corresponding XML item fields | Convert to numeric; calculate `round(quantity * unit_price, 2)` | Recompute from canonical item fields rather than choosing one source total |

Do not copy the fictional paths into your submitted mapping. Investigate your
allocated files and cite your own notebook evidence.


In [14]:
# Optional display helper. Adjust the path if you place the template elsewhere.
import pandas as pd

mapping_template_path = TEMPLATE_DIR / "A1_source_to_target_mapping_template.csv"
mapping = pd.read_csv(mapping_template_path, keep_default_na=False)
mapping.head()


FileNotFoundError: [Errno 2] No such file or directory: 'templates/A1_source_to_target_mapping_template.csv'

## 3. Text and regex functions

Implement and test the six required functions in
`GroupNNN_text_functions.py`. Show public cases plus your own matched,
unmatched, missing, multilingual and near-match cases here.


### 3.1 Cleaning and extraction implementation


### 3.2 Public and student-designed tests


## 4. Build the six standardised relational tables

Show the transformation and row-flow evidence for each table. Keep helper
columns inside the workflow; export only fields in the public data dictionary.


### 4.1 `orders`


### 4.2 `order_items`


### 4.3 `customers`


### 4.4 `deliveries`


### 4.5 `products`


### 4.6 `product_reviews`


## 5. Reconcile overlap and verify relationships

Demonstrate how records are compared by stable business key, how canonical
rows are retained and how silent source-precedence choices are avoided.


## 6. Validation register

Keep each check executable and give it a stable `VAL-...` ID. Immediately after
each code check, record the observed result, `PASS`/`FAIL`, evidence and
resolution/interpretation. A genuine, explained failure is preferable to a
fabricated pass.

Required areas include schema/types, primary and foreign keys, row flow and
source coverage, overlap, arithmetic, temporal logic, text/reference behaviour
and multilingual handling.


### 6.1 Schema and type checks (`VAL-SCHEMA-...`)


**Observed result/status/interpretation:** Replace.


### 6.2 Primary- and foreign-key checks (`VAL-PK-...`, `VAL-FK-...`)


**Observed result/status/interpretation:** Replace.


### 6.3 Source coverage and reconciliation checks (`VAL-FLOW-...`)


**Observed result/status/interpretation:** Replace.


### 6.4 Arithmetic checks (`VAL-ARITH-...`)


**Observed result/status/interpretation:** Replace.


### 6.5 Temporal checks (`VAL-TIME-...`)


**Observed result/status/interpretation:** Replace.


### 6.6 Text and multilingual checks (`VAL-TEXT-...`)


**Observed result/status/interpretation:** Replace.


### 6.7 Literal `NaN` reminder

For prescribed missing string outputs, the expected value is the three text
characters `NaN`, not an empty field, Python `None` or a floating-point NaN.
Use `pandas.read_csv(path, keep_default_na=False)` when validating that sentinel.


## 7. Export the six CSV files

Export exactly the required filenames, columns and order. Display a compact
final schema/row-count summary without hard-coding certified counts.


## 8. Final reproducibility record

Record the final run date, dependency versions and the result of Restart and Run
All. Confirm that the six outputs and validation evidence were recreated from
the allocated raw files.
